# 12 — Full pipeline for a new image: Real / Full AI / Partial AI → where & how much

**Why this notebook exists:** notebook 11 only runs the *localization* model. That model was trained only on
partially-edited images, so it answers "where is the edited patch?" — it cannot say whether an image is real, fully AI or partially AI.
A real photo and a fully AI image both give ~0 % there (no edited patch *inside* an otherwise real photo).

**Pipeline here**
1. **Stage 1 – classify** with your own models: notebook-01 features (836 handcrafted + 768 DINOv2 → top-500 → z-score),
   your 4 scratch classifiers (LR, Linear SVM, RBF SVM, RF) and the **stacking** meta-model from notebook 09.
2. **Stage 2 – localize/quantify**: Partial AI → U-Net mask + area %; Full AI → whole image (100 %); Real → 0 %.
3. **Shortcut check:** in your training data every *real* image is a JPEG (COCO) and every *AI* image is a PNG.
   The last section re-saves AI test images as JPEG and measures whether the classifier then calls them "Real".

Runs on the laptop (GPU) or Colab (CPU is fine for single images).

In [ ]:
# ============================================================
# 1. Setup + paths
# ============================================================
import os, sys, io, json, pickle, subprocess, warnings
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
def ensure(module, pip_name=None):
    try: __import__(module)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or module])
ensure("transformers"); ensure("segmentation_models_pytorch", "segmentation-models-pytorch")
import numpy as np, pandas as pd, torch, torch.nn.functional as F, matplotlib.pyplot as plt
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import segmentation_models_pytorch as smp
from transformers import AutoImageProcessor, AutoModel
warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

PROJECT_DIR = (Path("/content/drive/MyDrive/AI_Image_Project") if IN_COLAB
               else Path(os.environ.get("AI_PROJECT_DIR", r"C:\AI_Image_Project")))   # <- laptop folder
FEATURES_DIR, MODELS_DIR = PROJECT_DIR / "features", PROJECT_DIR / "saved_models"
LOC_RUN = "Unet_resnet34_crop512_plusbbox_clean"          # final localization model
DINO_MODEL_NAME = os.environ.get("AI_DINO_MODEL", "facebook/dinov2-base")   # same backbone as notebook 01
CLASS_NAMES = ["Real (COCO-like photo)", "Full AI", "Partial AI"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

needed = [FEATURES_DIR / "top500_feature_indices.npy", FEATURES_DIR / "scaler_params.npz",
          FEATURES_DIR / "y_labels_15k.npy", FEATURES_DIR / "val_idx.npy",
          MODELS_DIR / "logreg_weights.npz", MODELS_DIR / "svm_model.npz", MODELS_DIR / "svm_rbf_model.npz",
          MODELS_DIR / "rf_scratch.pkl"] + [MODELS_DIR / f"{k}_val_probs.npy" for k in ("logreg", "rf_scratch", "svm_rbf", "svm")] \
         + [MODELS_DIR / "localization" / f"{LOC_RUN}_deploy.pth"]
missing = [str(p.relative_to(PROJECT_DIR)) for p in needed if not p.exists()]
assert not missing, "Download these from Drive into the same folders:\n  " + "\n  ".join(missing)
print("Device:", DEVICE, "| all model files found")

In [ ]:
# ============================================================
# 2. Notebook-01 handcrafted feature extractor (copied verbatim) + DINOv2
# ============================================================
IMAGE_SIZE = (128, 128)

# Precomputed radial coordinate maps for ultra-fast 2D FFT features
_Y, _X = np.ogrid[:128, :128]
_RADIUS_MAP = np.sqrt((_X - 64) ** 2 + (_Y - 64) ** 2)
_ANGLE_MAP = np.arctan2(_Y - 64, _X - 64)

def calculate_skew_kurtosis(arr):
    """Computes sample skewness and kurtosis in pure NumPy."""
    flat = arr.ravel()
    mean_val = np.mean(flat)
    std_val = np.std(flat) + 1e-8
    z = (flat - mean_val) / std_val
    skew = np.mean(z ** 3)
    kurt = np.mean(z ** 4) - 3.0
    return float(skew), float(kurt)

def extract_domain1_spatial(img_rgb, gray):
    """Global spatial channel stats and cross-channel correlations (36 features)."""
    feats = []
    channels = [img_rgb[:, :, 0], img_rgb[:, :, 1], img_rgb[:, :, 2], gray]
    for ch in channels:
        s, k = calculate_skew_kurtosis(ch)
        feats.extend([
            float(np.mean(ch)), float(np.std(ch)), float(np.var(ch)),
            float(np.min(ch)), float(np.max(ch)), float(np.median(ch)),
            s, k
        ])

    r_flat = img_rgb[:, :, 0].ravel()
    g_flat = img_rgb[:, :, 1].ravel()
    b_flat = img_rgb[:, :, 2].ravel()

    c_rg = np.corrcoef(r_flat, g_flat)[0, 1] if np.std(r_flat) > 1e-6 and np.std(g_flat) > 1e-6 else 0.0
    c_rb = np.corrcoef(r_flat, b_flat)[0, 1] if np.std(r_flat) > 1e-6 and np.std(b_flat) > 1e-6 else 0.0
    c_gb = np.corrcoef(g_flat, b_flat)[0, 1] if np.std(g_flat) > 1e-6 and np.std(b_flat) > 1e-6 else 0.0
    rg_diff_std = float(np.std(img_rgb[:, :, 0] - img_rgb[:, :, 1]))

    feats.extend([float(np.nan_to_num(c_rg)), float(np.nan_to_num(c_rb)), float(np.nan_to_num(c_gb)), rg_diff_std])
    return feats

def extract_domain2_histograms(img_rgb, gray):
    """Intensity histograms across RGB and grayscale (128 features)."""
    feats = []
    channels = [img_rgb[:, :, 0], img_rgb[:, :, 1], img_rgb[:, :, 2], gray]
    for ch in channels:
        hist, _ = np.histogram(ch, bins=32, range=(0.0, 1.0), density=False)
        hist = hist.astype(np.float32) / (ch.size + 1e-8)
        feats.extend(hist.tolist())
    return feats

def extract_domain3_fft(gray):
    """2D FFT spectral radial rings and angular wedges (64 features)."""
    feats = []
    fft_shift = np.fft.fftshift(np.fft.fft2(gray))
    mag = np.log1p(np.abs(fft_shift))

    # Global stats
    s, k = calculate_skew_kurtosis(mag)
    feats.extend([
        float(np.mean(mag)), float(np.std(mag)), float(np.max(mag)), float(np.min(mag)),
        float(np.percentile(mag, 25)), float(np.percentile(mag, 50)),
        float(np.percentile(mag, 75)), float(np.percentile(mag, 95))
    ])

    # 16 Concentric rings
    edges = np.linspace(0, 64.0, 17)
    for i in range(16):
        mask = (_RADIUS_MAP >= edges[i]) & (_RADIUS_MAP < edges[i + 1])
        vals = mag[mask]
        if len(vals) > 0:
            feats.extend([float(np.mean(vals)), float(np.std(vals)), float(np.max(vals))])
        else:
            feats.extend([0.0, 0.0, 0.0])

    # 4 Angular wedges
    ang_edges = np.linspace(-np.pi, np.pi, 5)
    for i in range(4):
        mask = (_ANGLE_MAP >= ang_edges[i]) & (_ANGLE_MAP < ang_edges[i + 1]) & (_RADIUS_MAP > 5)
        vals = mag[mask]
        feats.extend([float(np.mean(vals)), float(np.std(vals))])

    return feats

def extract_domain4_residuals(img_rgb, gray):
    """Sensor noise residuals and SRM spatial high-pass filters (68 features)."""
    feats = []
    smooth = (
        np.roll(gray, 1, axis=0) + np.roll(gray, -1, axis=0) +
        np.roll(gray, 1, axis=1) + np.roll(gray, -1, axis=1)
    ) / 4.0
    res_lap = gray - smooth
    res_h = gray - np.roll(gray, 1, axis=1)
    res_v = gray - np.roll(gray, 1, axis=0)
    res_d = (gray + np.roll(np.roll(gray, 1, 0), 1, 1)) - (np.roll(gray, 1, 0) + np.roll(gray, 1, 1))

    for r in [res_lap, res_h, res_v, res_d]:
        s, k = calculate_skew_kurtosis(r)
        abs_r = np.abs(r)
        feats.extend([
            float(np.mean(r)), float(np.std(r)), float(np.var(r)),
            float(np.mean(abs_r)), float(np.std(abs_r)),
            float(np.max(r)), float(np.min(r)), s, k,
            float(np.percentile(r, 5)), float(np.percentile(r, 25)),
            float(np.percentile(r, 50)), float(np.percentile(r, 75)),
            float(np.percentile(r, 95))
        ])

    for c in range(3):
        ch = img_rgb[:, :, c]
        ch_res = ch - (np.roll(ch, 1, 0) + np.roll(ch, -1, 0) + np.roll(ch, 1, 1) + np.roll(ch, -1, 1)) / 4.0
        feats.extend([
            float(np.mean(np.abs(ch_res))), float(np.std(ch_res)),
            float(np.percentile(ch_res, 90)), float(np.percentile(ch_res, 10))
        ])

    return feats, res_lap

def extract_domain5_gradients(gray):
    """Gradients and 4-bit Local Binary Pattern (LBP) histogram (60 features)."""
    feats = []
    gx = np.roll(gray, -1, axis=1) - np.roll(gray, 1, axis=1)
    gy = np.roll(gray, -1, axis=0) - np.roll(gray, 1, axis=0)
    g_mag = np.sqrt(gx ** 2 + gy ** 2)
    g_ori = np.arctan2(gy, gx)

    s, k = calculate_skew_kurtosis(g_mag)
    feats.extend([
        float(np.mean(g_mag)), float(np.std(g_mag)), float(np.var(g_mag)),
        float(np.max(g_mag)), float(np.median(g_mag)), s, k,
        float(np.percentile(g_mag, 10)), float(np.percentile(g_mag, 25)),
        float(np.percentile(g_mag, 75)), float(np.percentile(g_mag, 90)),
        float(np.percentile(g_mag, 99))
    ])

    h_mag, _ = np.histogram(g_mag, bins=16, range=(0.0, 1.5), density=False)
    feats.extend((h_mag.astype(np.float32) / (g_mag.size + 1e-8)).tolist())

    h_ori, _ = np.histogram(g_ori, bins=16, range=(-np.pi, np.pi), density=False)
    feats.extend((h_ori.astype(np.float32) / (g_ori.size + 1e-8)).tolist())

    # 4-bit LBP
    b0 = (gray > np.roll(gray, 1, 0)).astype(np.int32)
    b1 = (gray > np.roll(gray, -1, 0)).astype(np.int32) * 2
    b2 = (gray > np.roll(gray, 1, 1)).astype(np.int32) * 4
    b3 = (gray > np.roll(gray, -1, 1)).astype(np.int32) * 8
    lbp = b0 + b1 + b2 + b3
    lbp_h, _ = np.histogram(lbp, bins=16, range=(0, 16), density=False)
    feats.extend((lbp_h.astype(np.float32) / (gray.size + 1e-8)).tolist())

    return feats, g_mag

def extract_domain6_patches(img_rgb, gray, res_lap, g_mag):
    """16 Local 32x32 patches for detecting splicing/inpainting seams (480 features)."""
    feats = []
    p_h, p_w = 32, 32
    for r in range(4):
        for c in range(4):
            y0, y1 = r * p_h, (r + 1) * p_h
            x0, x1 = c * p_w, (c + 1) * p_w

            p_rgb = img_rgb[y0:y1, x0:x1]
            p_gray = gray[y0:y1, x0:x1]
            p_res = res_lap[y0:y1, x0:x1]
            p_grad = g_mag[y0:y1, x0:x1]

            # Patch color
            for ch in range(3):
                feats.extend([float(np.mean(p_rgb[:, :, ch])), float(np.std(p_rgb[:, :, ch]))])
            feats.extend([float(np.mean(p_gray)), float(np.std(p_gray))])

            # Patch noise
            s, k = calculate_skew_kurtosis(p_res)
            abs_res = np.abs(p_res)
            feats.extend([
                float(np.mean(p_res)), float(np.std(p_res)), float(np.var(p_res)),
                float(np.mean(abs_res)), float(np.max(abs_res)), s, k,
                float(np.percentile(p_res, 10)), float(np.percentile(p_res, 50)),
                float(np.percentile(p_res, 90))
            ])

            # Patch gradient
            feats.extend([
                float(np.mean(p_grad)), float(np.std(p_grad)), float(np.max(p_grad)),
                float(np.percentile(p_grad, 25)), float(np.percentile(p_grad, 50)),
                float(np.percentile(p_grad, 90))
            ])

            # Patch spectral
            p_fft = np.fft.fftshift(np.fft.fft2(p_gray))
            p_mag = np.log1p(np.abs(p_fft))
            low_f = p_mag[12:20, 12:20]
            high_f = p_mag[:8, :8]
            feats.extend([
                float(np.mean(p_mag)), float(np.std(p_mag)),
                float(np.mean(low_f)), float(np.std(low_f)),
                float(np.mean(high_f)), float(np.std(high_f))
            ])

    return feats

def extract_all_handcrafted_features(image_path):
    """Master single-image extractor returning a clean (836,) vector."""
    try:
        with Image.open(image_path) as img:
            rgb = img.convert("RGB").resize(IMAGE_SIZE, Image.BILINEAR)
            arr = np.asarray(rgb, dtype=np.float32) / 255.0

        gray = 0.299 * arr[:, :, 0] + 0.587 * arr[:, :, 1] + 0.114 * arr[:, :, 2]

        f1 = extract_domain1_spatial(arr, gray)
        f2 = extract_domain2_histograms(arr, gray)
        f3 = extract_domain3_fft(gray)
        f4, res_lap = extract_domain4_residuals(arr, gray)
        f5, g_mag = extract_domain5_gradients(gray)
        f6 = extract_domain6_patches(arr, gray, res_lap, g_mag)

        vector = f1 + f2 + f3 + f4 + f5 + f6
        return np.nan_to_num(np.array(vector, dtype=np.float32), nan=0.0)
    except Exception:
        return np.zeros(836, dtype=np.float32)

print("836-Feature Forensic Extractor compiled successfully.")

def handcrafted_from_pil(img):
    # Same as extract_all_handcrafted_features(), but from an already-open image (so we can test JPEG re-saving in memory)
    rgb = img.convert("RGB").resize(IMAGE_SIZE, Image.BILINEAR)
    arr = np.asarray(rgb, dtype=np.float32) / 255.0
    gray = 0.299 * arr[:, :, 0] + 0.587 * arr[:, :, 1] + 0.114 * arr[:, :, 2]
    f1 = extract_domain1_spatial(arr, gray); f2 = extract_domain2_histograms(arr, gray); f3 = extract_domain3_fft(gray)
    f4, res_lap = extract_domain4_residuals(arr, gray); f5, g_mag = extract_domain5_gradients(gray)
    f6 = extract_domain6_patches(arr, gray, res_lap, g_mag)
    return np.nan_to_num(np.array(f1 + f2 + f3 + f4 + f5 + f6, dtype=np.float32), nan=0.0)

processor = AutoImageProcessor.from_pretrained(DINO_MODEL_NAME)
dino = AutoModel.from_pretrained(DINO_MODEL_NAME).to(DEVICE).eval()

@torch.no_grad()
def dino_from_pil(img):
    px = processor(images=img.convert("RGB"), return_tensors="pt")["pixel_values"].to(DEVICE)
    with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
        out = dino(pixel_values=px)
    return out.last_hidden_state[:, 0, :].float().cpu().numpy()[0]

TOP500 = np.load(FEATURES_DIR / "top500_feature_indices.npy")
_sc = np.load(FEATURES_DIR / "scaler_params.npz"); MU, SIGMA = _sc["mu"], _sc["sigma"]

def features_from_pil(img):
    x = np.concatenate([handcrafted_from_pil(img), dino_from_pil(img)])[TOP500]
    return ((x - MU) / (SIGMA + 1e-8))[None, :]          # (1, 500), same normalisation as notebooks 02-05
print("Feature pipeline ready:", len(TOP500), "selected features")

In [ ]:
# ============================================================
# 3. Your 4 classifiers (from saved files) + the stacking meta-model of notebook 09
# ============================================================
# Random-forest classes from notebook 05 (needed to load rf_scratch.pkl)
from sklearn.metrics import accuracy_score
class Node:
    def __init__(self, predicted_class, depth):
        self.predicted_class = predicted_class
        self.depth = depth
        self.feature_index = 0
        self.threshold = 0.0
        self.left = None
        self.right = None
        self.is_leaf = True

class DecisionTreeScratch:
    def __init__(self, max_depth=10, min_samples_leaf=1, max_features=None, n_bins=32):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_bins = n_bins
        self.tree = None
        self.feature_importances_ = None
        self.n_classes = None

    def fit(self, X, y):
        self.n_classes = len(np.unique(y))
        self.feature_importances_ = np.zeros(X.shape[1])
        self.tree = self._grow_tree(X, y, depth=0)
        return self

    def _gini(self, y):
        m = len(y)
        if m == 0: return 0
        counts = np.bincount(y, minlength=self.n_classes)
        probs = counts / m
        return 1.0 - np.sum(probs ** 2)

    def _grow_tree(self, X, y, depth):
        num_samples, num_features = X.shape
        counts = np.bincount(y, minlength=self.n_classes)
        predicted_class = np.argmax(counts)
        node = Node(predicted_class=predicted_class, depth=depth)

        # Stop criteria
        if depth >= self.max_depth or num_samples < self.min_samples_leaf * 2 or len(np.unique(y)) == 1:
            return node

        # Random feature subset
        feat_idxs = np.random.choice(num_features, self.max_features, replace=False)

        best_gini = float('inf')
        best_split = None

        # Fast split search using binning
        for feat in feat_idxs:
            X_col = X[:, feat]
            # Create quantile-based thresholds to speed up search
            thresholds = np.percentile(X_col, np.linspace(5, 95, self.n_bins))
            thresholds = np.unique(thresholds)

            for threshold in thresholds:
                left_mask = X_col <= threshold
                right_mask = ~left_mask

                if np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf:
                    continue

                gini_left = self._gini(y[left_mask])
                gini_right = self._gini(y[right_mask])
                n_l, n_r = np.sum(left_mask), np.sum(right_mask)
                gini_mix = (n_l * gini_left + n_r * gini_right) / num_samples

                if gini_mix < best_gini:
                    best_gini = gini_mix
                    best_split = (feat, threshold, left_mask, right_mask)

        if best_split is not None:
            feat, threshold, left_mask, right_mask = best_split
            node.feature_index = feat
            node.threshold = threshold
            node.is_leaf = False

            # Feature importance tracking (impurity decrease)
            parent_gini = self._gini(y)
            self.feature_importances_[feat] += (parent_gini - best_gini) * num_samples

            node.left = self._grow_tree(X[left_mask], y[left_mask], depth + 1)
            node.right = self._grow_tree(X[right_mask], y[right_mask], depth + 1)

        return node

    def _predict_row(self, row, node):
        if node.is_leaf:
            return node.predicted_class
        if row[node.feature_index] <= node.threshold:
            return self._predict_row(row, node.left)
        return self._predict_row(row, node.right)

    def predict(self, X):
        return np.array([self._predict_row(row, self.tree) for row in X])


class RandomForestScratch:
    def __init__(self, n_estimators=100, max_depth=10, min_samples_leaf=1, max_features='sqrt', random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.max_features_str = max_features
        self.random_state = random_state
        self.trees = []
        self.feature_importances_ = None
        self.oob_score_ = None
        self.n_classes = None

    def fit(self, X, y):
        np.random.seed(self.random_state)
        self.n_classes = len(np.unique(y))
        n_samples, n_features = X.shape

        if self.max_features_str == 'sqrt':
            self.max_features = int(np.sqrt(n_features))
        elif self.max_features_str == 'log2':
            self.max_features = int(np.log2(n_features))
        else:
            self.max_features = int(n_features * 0.5)

        self.trees = []
        self.feature_importances_ = np.zeros(n_features)
        oob_predictions = np.zeros((n_samples, self.n_classes))
        oob_counts = np.zeros(n_samples)

        print(f"Training {self.n_estimators} trees...")
        for i in range(self.n_estimators):
            # Bootstrap sampling
            indices = np.random.choice(n_samples, n_samples, replace=True)
            oob_indices = np.setdiff1d(np.arange(n_samples), indices)

            tree = DecisionTreeScratch(max_depth=self.max_depth, min_samples_leaf=self.min_samples_leaf, max_features=self.max_features,n_bins=16)
            tree.fit(X[indices], y[indices])
            self.trees.append(tree)
            self.feature_importances_ += tree.feature_importances_

            # OOB predictions
            if len(oob_indices) > 0:
                preds = tree.predict(X[oob_indices])
                for idx, p in zip(oob_indices, preds):
                    oob_predictions[idx, p] += 1
                    oob_counts[idx] += 1

            if (i + 1) % 10 == 0:
                print(f"Tree {i + 1}/{self.n_estimators} built.")

        # Normalize feature importances
        if np.sum(self.feature_importances_) > 0:
            self.feature_importances_ /= np.sum(self.feature_importances_)

        # Calculate OOB score
        valid_oob = oob_counts > 0
        if np.any(valid_oob):
            oob_final_preds = np.argmax(oob_predictions[valid_oob], axis=1)
            self.oob_score_ = accuracy_score(y[valid_oob], oob_final_preds)

        return self

    def predict_proba(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees]) # Shape: (n_estimators, n_samples)
        probs = np.zeros((X.shape[0], self.n_classes))
        for i in range(X.shape[0]):
            counts = np.bincount(tree_preds[:, i], minlength=self.n_classes)
            probs[i] = counts / self.n_estimators
        return probs

    def predict(self, X):
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)

def _softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def _svm_pseudo_proba(scores):            # exactly the predict_proba of notebooks 03/04
    s = 1.0 / (1.0 + np.exp(-np.clip(scores, -50, 50))); return s / s.sum(1, keepdims=True)

LR = np.load(MODELS_DIR / "logreg_weights.npz"); SVM = np.load(MODELS_DIR / "svm_model.npz", allow_pickle=True)
RBF = np.load(MODELS_DIR / "svm_rbf_model.npz", allow_pickle=True)
with open(MODELS_DIR / "rf_scratch.pkl", "rb") as f:
    RF = pickle.load(f)

def base_probs(X):
    D = RBF["rff_W"].shape[1]
    mapped = np.sqrt(2.0 / D) * np.cos(X @ RBF["rff_W"] + RBF["rff_b"])
    return {"logreg": _softmax(X @ LR["W"] + LR["b"]),
            "rf_scratch": RF.predict_proba(X),
            "svm_rbf": _svm_pseudo_proba(mapped @ RBF["W"].T + RBF["b"]),
            "svm": _svm_pseudo_proba(X @ SVM["W"].T + SVM["b"])}
BASE_ORDER = ["logreg", "rf_scratch", "svm_rbf", "svm"]
NICE = {"logreg": "Logistic Regression", "rf_scratch": "Random Forest", "svm_rbf": "RBF SVM", "svm": "Linear SVM"}

def stack_input(P):
    return np.log(np.concatenate([np.clip(P[k], 1e-7, 1) / np.clip(P[k], 1e-7, 1).sum(1, keepdims=True) for k in BASE_ORDER], 1))

# Stacking meta-model: same recipe as notebook 09, fitted on the saved VALIDATION probabilities
y_all = np.load(FEATURES_DIR / "y_labels_15k.npy", allow_pickle=True); val_idx = np.load(FEATURES_DIR / "val_idx.npy")
P_val = {k: np.load(MODELS_DIR / f"{k}_val_probs.npy").astype(float) for k in BASE_ORDER}
META = LogisticRegression(C=0.5, max_iter=5000).fit(stack_input(P_val), y_all[val_idx])
print("Stacking meta-model fitted on", len(val_idx), "validation images")

# Optional self-check: recomputed probabilities from the saved models must match the saved ones
Xc = FEATURES_DIR / "X_combined_500.npy"
if Xc.exists():
    Xv = (np.load(Xc, mmap_mode="r")[val_idx[:25]] - MU) / (SIGMA + 1e-8)
    re = base_probs(np.asarray(Xv))
    for k in BASE_ORDER:
        print(f"  check {NICE[k]:20s}: max |recomputed - saved| = {np.abs(re[k] - P_val[k][:25]).max():.2e}")

In [ ]:
# ============================================================
# 4. Stage 2 model: final localization U-Net (same as notebook 11)
# ============================================================
ck = torch.load(MODELS_DIR / "localization" / f"{LOC_RUN}_deploy.pth", map_location="cpu", weights_only=False)
LCFG = ck["cfg"]
unet = smp.create_model(arch=LCFG["ARCH"], encoder_name=LCFG["ENCODER"], encoder_weights=None, in_channels=3, classes=1)
unet.load_state_dict(ck["model"]); unet.to(DEVICE).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1) * 255
STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1) * 255

@torch.no_grad()
def localize(img):
    s = min(1.0, LCFG["MAX_SIDE"] / max(img.size))
    if s < 1: img = img.resize((round(img.size[0] * s), round(img.size[1] * s)), Image.Resampling.BICUBIC)
    a = np.asarray(img.convert("RGB")); H, W = a.shape[:2]
    x = (torch.from_numpy(a.copy()).to(DEVICE).permute(2, 0, 1)[None].float() - MEAN) / STD
    x = F.pad(x, (0, (-W) % 32, 0, (-H) % 32), mode="reflect")
    with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        p = torch.sigmoid(unet(x).float()); p = 0.5 * (p + torch.sigmoid(unet(x.flip(-1)).float()).flip(-1))
    prob = p[0, 0, :H, :W].float().cpu().numpy()
    mask = prob >= ck["threshold"]
    est = ck["quant_estimator"]; hard, soft = 100 * mask.mean(), 100 * prob.mean()
    if est.startswith("Hard mask @MAE"):    area = 100 * (prob >= ck["mae_threshold"]).mean()
    elif est.startswith("Hard mask"):       area = hard
    elif est.startswith("Soft + isotonic"): area = float(np.interp(soft, *ck["iso_soft"]))
    elif est.startswith("Hard + isotonic"): area = float(np.interp(hard, *ck["iso_hard"]))
    else:                                   area = soft
    return a, prob, mask, area
print("Localization model:", LOC_RUN)

In [ ]:
# ============================================================
# 5. analyze(path): Stage 1 classify -> Stage 2 where / how much
# ============================================================
def classify_pil(img):
    P = base_probs(features_from_pil(img))
    stacked = META.predict_proba(stack_input(P))[0]
    return stacked, {NICE[k]: P[k][0] for k in BASE_ORDER}

def analyze(path, show=True):
    img = Image.open(path).convert("RGB")
    probs, per_model = classify_pil(img)
    cls = int(np.argmax(probs)); conf = float(probs[cls])
    a, prob_map, mask, loc_area = localize(img)
    if cls == 2:   area, note = loc_area, "Partial AI -> edited region from the localization model"
    elif cls == 1: area, note = 100.0, "Full AI -> the whole image is generated"
    else:          area, note = 0.0, "Real -> no AI content expected"

    print(f"\n{Path(path).name}")
    print("  Stage 1 (stacking):  " + " | ".join(f"{n.split(' (')[0]} {p*100:.1f}%" for n, p in zip(CLASS_NAMES, probs)))
    print("  Individual models:   " + " | ".join(f"{m}: {CLASS_NAMES[int(np.argmax(p))].split(' (')[0]}" for m, p in per_model.items()))
    print(f"  Decision: {CLASS_NAMES[cls]}  (confidence {conf*100:.0f}%)  ->  AI area = {area:.2f}%   [{note}]")
    if conf < 0.6:
        print("  NOTE: low confidence - the image may be unlike the training data (see the shortcut check below).")
    if cls != 2 and loc_area > 2:
        print(f"  NOTE: the localization model alone would mark {loc_area:.1f}% as edited.")
    if show:
        ov = a.copy(); ov[mask] = (0.5 * ov[mask] + [127, 0, 0]).astype(np.uint8)
        fig, ax = plt.subplots(1, 4, figsize=(20, 5), gridspec_kw={"width_ratios": [1, 1, 1, 0.8]})
        ax[0].imshow(a); ax[0].set_title("input")
        ax[1].imshow(prob_map, cmap="jet", vmin=0, vmax=1); ax[1].set_title("localization probability")
        ax[2].imshow(ov); ax[2].set_title(f"edited region (used only if Partial AI): {loc_area:.2f}%")
        for x_ in ax[:3]: x_.axis("off")
        ax[3].barh([n.split(" (")[0] for n in CLASS_NAMES], probs * 100, color=["#2ca02c", "#d62728", "#ff7f0e"])
        ax[3].set_xlim(0, 100); ax[3].set_xlabel("probability (%)"); ax[3].set_title(f"{CLASS_NAMES[cls].split(' (')[0]} -> AI area {area:.1f}%")
        plt.suptitle(Path(path).name); plt.tight_layout(); plt.show()
    return {"class": CLASS_NAMES[cls], "probs": probs, "ai_area_pct": area, "localization_area_pct": loc_area}

In [ ]:
# ============================================================
# 6. Your images
# ============================================================
result = analyze(r"C:\Users\gopic\Downloads\firstimg.jpeg")
# for f in Path(r"C:\Users\gopic\Downloads\test_images").glob("*.*"): analyze(f)

In [ ]:
# ============================================================
# 7. Shortcut check: does the classifier rely on JPEG vs PNG?
#    Training data: Real = COCO JPEGs, Full AI / Partial AI = PNGs. We take AI TEST images, re-save them as JPEG
#    (quality 90, like a phone or WhatsApp would) and see how often the prediction flips to "Real".
# ============================================================
N_PER_CLASS = 40
paths_all = np.load(FEATURES_DIR / "image_paths_15k.npy", allow_pickle=True)
test_idx = np.load(FEATURES_DIR / "test_idx.npy")
def local_path(p):
    p = str(p).replace("\\", "/"); key = "AI_Image_Project/"
    return PROJECT_DIR / p.split(key, 1)[1] if key in p else Path(p)
def as_jpeg(img, q=90):
    buf = io.BytesIO(); img.convert("RGB").save(buf, "JPEG", quality=q); buf.seek(0); return Image.open(buf).convert("RGB")

rng = np.random.default_rng(0); rows = []
for c in range(3):
    idx = [i for i in test_idx[y_all[test_idx] == c] if local_path(paths_all[i]).exists()]
    if not idx:
        print(f"{CLASS_NAMES[c]}: no test images found locally (folder not downloaded) - skipped"); continue
    idx = rng.choice(idx, size=min(N_PER_CLASS, len(idx)), replace=False)
    for i in idx:
        img = Image.open(local_path(paths_all[i])).convert("RGB")
        rows.append({"true": c,
                     "pred_original": int(np.argmax(classify_pil(img)[0])),
                     "pred_jpeg90": int(np.argmax(classify_pil(as_jpeg(img, 90))[0])),
                     "pred_jpeg75": int(np.argmax(classify_pil(as_jpeg(img, 75))[0]))})
chk = pd.DataFrame(rows)
if len(chk):
    srows = []
    for c, d in chk.groupby("true"):
        srows.append({"class": CLASS_NAMES[c], "images": len(d),
                      "accuracy original": (d["pred_original"] == c).mean(),
                      "accuracy after JPEG q90": (d["pred_jpeg90"] == c).mean(),
                      "accuracy after JPEG q75": (d["pred_jpeg75"] == c).mean(),
                      "called Real (original)": (d["pred_original"] == 0).mean(),
                      "called Real after JPEG q90": (d["pred_jpeg90"] == 0).mean()})
    summary = pd.DataFrame(srows).set_index("class")
    print(summary.round(3).to_string())
    print("\nIf AI images flip to 'Real' after JPEG saving, the classifier learned the file format, not AI content.")

In [ ]:
# ============================================================
# 8. Feature check: do features computed here match the ones notebook 01 saved?
#    (needs features/X_combined_500.npy, ~30 MB). If they differ, every new image gets slightly wrong features.
# ============================================================
X500_file = FEATURES_DIR / "X_combined_500.npy"
if not X500_file.exists():
    print("Download features/X_combined_500.npy from Drive to run this check.")
else:
    X500 = np.load(X500_file, mmap_mode="r")
    hand = TOP500 < 836                                   # which of the 500 features are handcrafted vs DINOv2
    pool = [i for i in test_idx if local_path(paths_all[i]).exists()]
    pick = np.random.default_rng(1).choice(pool, size=min(40, len(pool)), replace=False)
    rows = []
    for i in pick:
        new = features_from_pil(Image.open(local_path(paths_all[i])).convert("RGB"))[0]
        old = (np.asarray(X500[i]) - MU) / (SIGMA + 1e-8)
        p_new = META.predict_proba(stack_input(base_probs(new[None])))[0]
        p_old = META.predict_proba(stack_input(base_probs(old[None])))[0]
        rows.append({"true": int(y_all[i]),
                     "corr_handcrafted": np.corrcoef(new[hand], old[hand])[0, 1],
                     "corr_dino": np.corrcoef(new[~hand], old[~hand])[0, 1],
                     "max_abs_diff": np.abs(new - old).max(),
                     "pred_saved_features": int(np.argmax(p_old)), "pred_recomputed": int(np.argmax(p_new))})
    fc = pd.DataFrame(rows)
    print(f"Images checked: {len(fc)}")
    print(f"Correlation recomputed vs saved  - handcrafted: median {fc.corr_handcrafted.median():.4f} (min {fc.corr_handcrafted.min():.4f})"
          f" | DINOv2: median {fc.corr_dino.median():.4f} (min {fc.corr_dino.min():.4f})")
    print(f"Accuracy with SAVED features (notebook 01):  {(fc.pred_saved_features == fc.true).mean():.3f}")
    print(f"Accuracy with RECOMPUTED features (here):    {(fc.pred_recomputed == fc.true).mean():.3f}")
    print(f"Same prediction both ways: {(fc.pred_saved_features == fc.pred_recomputed).mean()*100:.0f}% of images")
    print("\nCorrelations near 1.000 and equal accuracies = pipeline matches notebook 01."
          "\nLow DINOv2 correlation = preprocessing differs from notebook 01 (library version) -> tell me the numbers.")